In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from sklearn import datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as tt

**DISCLAIMER**

The presented code is not optimized, it serves an educational purpose. It is written for CPU, it uses only fully-connected networks and an extremely simplistic dataset. However, it contains all components that can help to understand how a diffusion denoise probabilistic model (DDPM) works, and it should be rather easy to extend it to more sophisticated models. This code could be run almost on any laptop/PC, and it takes a couple of minutes top to get the result.

## Dataset: Digits

In this example, we go wild and use a dataset that is simpler than MNIST! We use a scipy dataset called Digits. It consists of ~1500 images of size 8x8, and each pixel can take values in $\{0, 1, \ldots, 16\}$.

The goal of using this dataset is that everyone can run it on a laptop, without any gpu etc.

In [2]:
class Digits(Dataset):
    """Scikit-Learn Digits dataset."""

    def __init__(self, mode='train', transforms=None):
        digits = load_digits()
        if mode == 'train':
            # We set up the training dataset
            self.data = digits.data[:1000].astype(np.float32)
        elif mode == 'val':
            # we set up the validation dataset
            self.data = digits.data[1000:1350].astype(np.float32)
        else:
            # otherwise, we set up the test dataset
            self.data = digits.data[1350:].astype(np.float32)

        # sometimes, we need transforms to transform the dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        if self.transforms:
            sample = self.transforms(sample)
        return sample

## Auxiliary functions and classes

### Distributions

these are the little log-likelihood helpers you’ll see all over diffusion (and generative) code. Here’s what each one is doing and why it’s written this way.

---

# Constants

* `PI = torch.from_numpy(np.asarray(np.pi))`: makes π a Torch scalar so it broadcasts and stays on the right device/dtype.
* `EPS = 1e-7`: tiny floor/ceiling to avoid `log(0)` and `log(1)` edge cases.

---

# 1) `log_categorical(x, p, num_classes=256, reduction=None, dim=None)`

**What it computes:** the log PMF of a categorical distribution for discrete targets `x` given class probabilities `p`.

**How it works:**

* `x_one_hot = F.one_hot(x.long(), num_classes)`: turn integer labels into one-hot (…, C).
* `torch.log(torch.clamp(p, EPS, 1-EPS))`: safe log of probs.
* `x_one_hot * log_p`: zeroes out all classes except the target class → you get a tensor that is `log p[target]` at the target index and `0` elsewhere.
* You’ll typically **sum over the class dimension** (`dim=class_dim`) to collapse to just `log p[target]` per item; otherwise you keep the sparse one-hot-shaped tensor.
* `reduction`: `'avg'` or `'sum'` lets you aggregate over `dim` (e.g., batch or spatial dims).

**Gotchas:** Make sure `dim` is the class axis if you want a scalar per sample; also ensure `p` is already normalized (softmaxed).

---

# 2) `log_bernoulli(x, p, reduction=None, dim=None)`

**What it computes:** the log PMF for Bernoulli variables with probabilities `p`, for targets `x ∈ {0,1}` (broadcasts work).

**Formula:**
`log P(x|p) = x * log p + (1 - x) * log(1 - p)` with `p` clamped to `(EPS, 1-EPS)`.

**When used:** binary pixels/masks, or binary latent indicators. (If you have logits instead of probs, you’d usually write a numerically stable version with `logits`.)

---

# 3) `log_normal_diag(x, mu, log_var, reduction=None, dim=None)`

**What it computes:** elementwise log-density of a diagonal Gaussian `N(mu, diag(exp(log_var)))`.

**Formula (per dimension):**
`log p(x) = -0.5*log(2π) - 0.5*log_var - 0.5*exp(-log_var)*(x - mu)^2`

**Why this form:**

* Uses `log_var` (a common parameterization) for stability and unconstrained optimization.
* Avoids building covariance matrices; everything broadcasts.

**Use in diffusion:** KLs and ELBO bits often involve normals; this is the workhorse for those terms.

---

# 4) `log_standard_normal(x, reduction=None, dim=None)`

**What it computes:** log-density of `N(0, I)`.

**Formula:**
`log p(x) = -0.5*log(2π) - 0.5*x^2`

**Use in diffusion:** priors, KL to standard normal, etc.

---

# 5) `log_min_exp(a, b, epsilon=1e-8)`

**What it computes:** a **stable** `log( exp(a) - exp(b) )`, assuming `b < a`.

**Derivation:**
`log(exp(a) - exp(b)) = a + log(1 - exp(b - a))`
When `a ≈ b`, `exp(b - a)` is close to 1 and naive subtraction is unstable; this form keeps numbers well-behaved. The tiny `epsilon` protects the `log(1 - ·)` from hitting exactly 0.

**Why needed:** next function computes a difference of CDFs (probability mass in a bin), which is `exp(a) - exp(b)` in probability space; doing it in log-space is safer.

---

# 6) `log_integer_probability(x, mean, logscale)`

**What it computes:** the **log probability mass** that a *continuous logistic* variable falls inside the **integer bin** `[x-0.5, x+0.5]`, i.e., the discretized logistic likelihood used for integer-valued data (e.g., 8-bit pixels).

**How it works:**

* Let `scale = exp(logscale)`. For a logistic with `(mean, scale)`, the CDF is `σ((t - mean)/scale)`.
* Probability that `t ∈ [x-0.5, x+0.5]` is:
  `σ((x+0.5 - mean)/scale) - σ((x-0.5 - mean)/scale)`.
* In **log space** that’s:
  `logσ(u1) ⟂minus⟂ logσ(u2)` → implemented as
  `log_min_exp( logσ(u1), logσ(u2) )`, with `u1 > u2` so the assumption `b < a` holds.

**Why this matters in DDPM-ish pipelines:** even though “vanilla” DDPMs typically use Gaussian objectives, many image likelihoods (PixelCNN, VAEs on images, or hybrid objectives) model discrete pixels via a **discretized logistic**; this helper gives you stable per-pixel log-PMFs.

---

# 7) `log_integer_probability_standard(x)`

**What it computes:** same as above but for a **standard logistic** (`mean=0`, `scale=1`).
Uses `log_min_exp(F.logsigmoid(x+0.5), F.logsigmoid(x-0.5))`.


In [3]:
PI = torch.from_numpy(np.asarray(np.pi))
EPS = 1.e-7

def log_categorical(x, p, num_classes=256, reduction=None, dim=None):

    # we define the log pmf values for categorical distribution    
    x_one_hot = F.one_hot(x.long(), num_classes=num_classes)
    log_p = x_one_hot * torch.log(torch.clamp(p, EPS, 1. - EPS))
    if reduction == 'avg':
        return torch.mean(log_p, dim)
    elif reduction == 'sum':
        return torch.sum(log_p, dim)
    else:
        return log_p

def log_bernoulli(x, p, reduction=None, dim=None):

    # we define the log pmf values for the Bernoulli distribution
    pp = torch.clamp(p, EPS, 1. - EPS)
    log_p = x * torch.log(pp) + (1. - x) * torch.log(1. - pp)
    if reduction == 'avg':
        return torch.mean(log_p, dim)
    elif reduction == 'sum':
        return torch.sum(log_p, dim)
    else:
        return log_p

def log_normal_diag(x, mu, log_var, reduction=None, dim=None):
    
    # we define the log pmf values for the multivariate normal distribution with the covariance matrix being the diagonal matrix
    log_p = -0.5 * torch.log(2. * PI) - 0.5 * log_var - 0.5 * torch.exp(-log_var) * (x - mu)**2.
    if reduction == 'avg':
        return torch.mean(log_p, dim)
    elif reduction == 'sum':
        return torch.sum(log_p, dim)
    else:
        return log_p

def log_standard_normal(x, reduction=None, dim=None):
    
    # we define the uni-variate standard normal distribution
    log_p = -0.5 * torch.log(2. * PI) - 0.5 * x**2.
    if reduction == 'avg':
        return torch.mean(log_p, dim)
    elif reduction == 'sum':
        return torch.sum(log_p, dim)
    else:
        return log_p

# Chakraborty & Chakravarty, "A new discrete probability distribution with integer support on (−∞, ∞)",
#  Communications in Statistics - Theory and Methods, 45:2, 492-505, DOI: 10.1080/03610926.2013.830743

def log_min_exp(a, b, epsilon=1e-8):
    """
    Source: https://github.com/jornpeters/integer_discrete_flows
    Computes the log of exp(a) - exp(b) in a (more) numerically stable fashion.
    Using:
    log(exp(a) - exp(b))
    c + log(exp(a-c) - exp(b-c))
    a + log(1 - exp(b-a))
    And note that we assume b < a always.
    """
    y = a + torch.log(1 - torch.exp(b - a) + epsilon)

    return y

def log_integer_probability(x, mean, logscale):
    scale = torch.exp(logscale)

    logp = log_min_exp(
      F.logsigmoid((x + 0.5 - mean) / scale),
      F.logsigmoid((x - 0.5 - mean) / scale))

    return logp

def log_integer_probability_standard(x):
    logp = log_min_exp(
      F.logsigmoid(x + 0.5),
      F.logsigmoid(x - 0.5))

    return logp

## Denoise Diffusion Probabilistc Models

**NOTE** Please note that we use a single class unlike in previous implementations of VAEs. We do it for clarity.

This class mixes **diffusion-style forward noising** with a **VAE-style ELBO** whose latent chain is the diffusion trajectory $z_1,\dots,z_T$. The forward process $q(z_t\mid z_{t-1})$ is a fixed Gaussian with scalar variance $\beta$. The reverse process $p_\theta(z_{t-1}\mid z_t)$ is parameterized by a **list of neural nets** (`p_dnns`), one per step, that predict $(\mu_t, \log\sigma_t^2)$. A simple decoder maps (something close to) $z_0$ to the data $x$.

In standard DDPMs you usually have:

* a **time-conditioned U-Net** (one network reused at all steps),
* a **noise schedule** $\{\beta_t\}_{t=1}^T$ with **cumulative alphas** $\bar\alpha_t$,
* an objective that predicts $\epsilon$, $x_0$, or $v$.

This code uses:

* a **list of nets** (`p_dnns[i]`) instead of a single time-conditioned net,
* a **single scalar** `beta` shared across all steps,
* an ELBO written explicitly as reconstruction minus KLs between forward and learned reverse Gaussians.

---

# Fields / init

* `p_dnns`: list of modules; each outputs **twice the latent dimensionality** to split into `mu_i` and `log_var_i`. Intended as $p_\theta(z_{t-1}\mid z_t)$ heads.
* `decoder_net`: maps the final latent to image space; defines $p_\theta(x\mid z_0)$ as a **unit-variance Gaussian** (see RE below).
* `D`: latent dimensionality (per sample vector; if you had images you’d flatten channels*H*W or work per-pixel).
* `T`: number of diffusion steps.
* `beta`: stored as a length-1 FloatTensor; conceptually the **noise level per step**. (In production you’d register a **schedule** `beta_t` as a buffer and use $\alpha_t=1-\beta_t$, $\bar\alpha_t=\prod_{s\le t}\alpha_s$.)

---

# Reparameterization helpers

* `reparameterization(mu, log_var)`: classic VAE trick
  $z = \mu + \sigma\odot\epsilon$, with $\sigma=\exp(0.5\log\sigma^2)$, $\epsilon\sim\mathcal N(0,I)$.

* `reparameterization_gaussian_diffusion(x, i)`: the **forward diffusion step**

  $$
  z_{t}=\sqrt{1-\beta}\,z_{t-1} + \sqrt{\beta}\,\epsilon.
  $$

  The `i` arg is currently unused; all steps use the same $\beta$.

> Note: standard DDPM forward noising usually uses **cumulative** coefficients
> $z_t=\sqrt{\bar\alpha_t}x+\sqrt{1-\bar\alpha_t}\,\epsilon$ for *direct* sampling from any $t$. This function composes small steps instead.

---

# forward(self, x, reduction='avg')

## 1) Forward (noising) pass

```python
zs = [self.reparameterization_gaussian_diffusion(x, 0)]
for i in range(1, self.T):
    zs.append(self.reparameterization_gaussian_diffusion(zs[-1], i))
```

* Creates a list `zs` of length **T**, where `zs[0]=z1`, …, `zs[T-1]=zT`.
  (There is **no explicit z0** stored; `x` itself is playing the role of $z_0$.)

## 2) Reverse (denoising) parameterization

```python
mus, log_vars = [], []
for i in range(len(self.p_dnns) - 1, -1, -1):
    h = self.p_dnns[i](zs[i+1])      # condition on z_{i+1}
    mu_i, log_var_i = torch.chunk(h, 2, dim=1)
    mus.append(mu_i)
    log_vars.append(log_var_i)
```

* Each reverse net conditions on **the next noisy state** $z_{i+1}$ and predicts parameters for $p_\theta(z_i\mid z_{i+1})$.
* Iterates from the last step down to the first, pushing into lists.

## 3) Decoder likelihood (reconstruction term)

```python
mu_x = self.decoder_net(zs[0])          # uses z1
RE = log_standard_normal(x - mu_x).sum(-1)
```

* Interprets $p_\theta(x\mid z_0)$ as $\mathcal N(\mu_x, I)$. Since $\log\mathcal N(x;\mu_x,I)=\log\mathcal N(x-\mu_x;0,I)$, `log_standard_normal(x - mu_x)` is the per-dimension log-likelihood.
* **But** it uses `zs[0] (= z1)` as the decoder input, which suggests $p_\theta(x\mid z_1)$, not $p_\theta(x\mid z_0)$. If you intend a canonical ladder $x \leftrightarrow z_0 \leftrightarrow z_1 \dots$, you likely want the decoder to take **`z` at step 0** (i.e., the variable right before adding any noise).

> Suggestion: either keep a `z0` explicitly (set `z0 = some function of x`, which here is just `z0 = x`) and decode from **`z0`**, or redefine your notation/comments to clarify you decode from `z1`.

## 4) KL terms

```python
KL = ( log_normal_diag(zs[-1], sqrt(1 - beta) * zs[-1], log(beta))
       - log_standard_normal(zs[-1]) ).sum(-1)
```

* This matches the ELBO’s boundary term at $t=T$:
  $\mathrm{KL}\big(q(z_T\mid z_{T-1}) \parallel p(z_T)\big)$, with $p(z_T)=\mathcal N(0,I)$. The forward mean at step $T$ (given $z_{T-1}$) under the code’s Markov step is $\sqrt{1-\beta}\,z_{T-1}$. But the code uses $\sqrt{1-\beta}\,z_T$ (note the **index mismatch**). Correctly, you’d need the **distribution of $z_T$ given $z_{T-1}$** evaluated **at the sampled $z_T$**, which depends on $z_{T-1}$, not $z_T$ itself.

Then for intermediate steps:

```python
for i in range(len(mus)):
    KL_i = ( log_normal_diag(zs[i], sqrt(1 - self.beta) * zs[i], log(self.beta))
             - log_normal_diag(zs[i], mus[i], log_vars[i]) ).sum(-1)
    KL = KL + KL_i
```

* Intended to sum $\mathrm{KL}\big(q(z_i\mid z_{i+1}) \parallel p_\theta(z_i\mid z_{i+1})\big)$ across steps.
* The **left** term uses the forward Gaussian with mean $\sqrt{1-\beta}\,z_{i}$ (again: should be $\sqrt{1-\beta}\,z_{i+1}$ because $q(\cdot \mid z_{i+1})$ predicts $z_i$ from $z_{i+1}$).
* The **right** term uses the network’s predicted mean/var conditioned on `zs[i+1]` (good), but due to the earlier **append ordering**, `mus[i]` typically corresponds to a **different step** than `zs[i]`.


## 5) Loss aggregation

```python
loss = -(RE - KL).mean()   # or sum
```

* Standard ELBO: maximize $ \text{RE} - \text{KL} \Rightarrow$ minimize negative of it.

---

# sample(self, batch\_size)

* Starts from $z_T \sim \mathcal N(0,I)$ (`torch.randn([batch_size, D])`).

* For $t=T,\dots,1$:

  1. compute `h = p_dnns[i](z_t)`, split into $\mu_t,\log\sigma_t^2$;
  2. sample $z_{t-1} = \mu_t + \sigma_t \odot \epsilon$.

* Finally decode `z` to `mu_x = decoder_net(z)`.


---

# sample\_diffusion(self, x)

* Just runs the forward noising repeatedly to produce a single `z_T` (white-noise-ish) from `x`. Useful for visualization/debugging.

---


In [5]:
class DDPM(nn.Module):
    
    # we define the DDPM class
    def __init__(self, p_dnns, decoder_net, beta, T, D):
        super(DDPM, self).__init__()

        # here p_dnns denotes the neural network structure that we are interested in
        # its definition is be given in the later parts
        self.p_dnns = p_dnns  # a list of sequentials

        # the decoder_network is the output layer, denoting how to transform the last layer to reconstruct the observasions (images in this case)
        self.decoder_net = decoder_net

        # other params
        self.D = D

        # T is the number of steps in DDPM
        self.T = T

        # beta is teh noise schedules
        self.beta = torch.FloatTensor([beta])

    # This reparameterization is almost the core trick in deep generative models
    @staticmethod
    def reparameterization(mu, log_var):
        # define the standard deviation
        # Note: we need to set log_var as the trainable parameter since we want log_var can be negative or positive
        std = torch.exp(0.5*log_var)
        # eps is defined to be sampled from standard Gaussian distribution, i.e., eps~N(0, 1)
        eps = torch.randn_like(std)
        
        # mu+std*eps ~ N(mu, std^2)
        return mu + std * eps

    def reparameterization_gaussian_diffusion(self, x, i):
        # This step corresponds to the forward step, corresponds to the equation in slide 5
        
        return torch.sqrt(1. - self.beta) * x + torch.sqrt(self.beta) * torch.randn_like(x)

    def forward(self, x, reduction='avg'):
        # =====
        # forward difussion
        zs = [self.reparameterization_gaussian_diffusion(x, 0)]

        # the forward diffusion will consist of T steps
        for i in range(1, self.T):
            zs.append(self.reparameterization_gaussian_diffusion(zs[-1], i))

        # =====
        # backward diffusion
        # in each step of the backward diffusion, we need to use neural networks to obtain mean and log_variance
        mus = []
        log_vars = []

        for i in range(len(self.p_dnns) - 1, -1, -1):
            # The backward diffusion is also proceeded in T steps, however, in a reverse order
            
            # self.p_ddns
            h = self.p_dnns[i](zs[i+1])

            # we split the output from p_dnns into two parts, one for mu_i and one for log_var_i
            mu_i, log_var_i = torch.chunk(h, 2, dim=1)
            mus.append(mu_i)
            log_vars.append(log_var_i)

        # we use the decoder_net to map the last layer output to the output mu_x
        mu_x = self.decoder_net(zs[0])

        # =====ELBO
        # Reconstrucction error
        RE = log_standard_normal(x - mu_x).sum(-1)

        # KL
        # The KL term required to regularize the objective function
        KL = (log_normal_diag(zs[-1], torch.sqrt(1. - self.beta) * zs[-1], torch.log(self.beta)) - log_standard_normal(zs[-1])).sum(-1)

        # We need an additional T KL terms to regularize the objective function
        for i in range(len(mus)):
            KL_i = (log_normal_diag(zs[i], torch.sqrt(1. - self.beta) * zs[i], torch.log(self.beta)) - log_normal_diag(zs[i], mus[i], log_vars[i])).sum(-1)

            KL = KL + KL_i

        # Define the final ELBO
        if reduction == 'sum':
            loss = -(RE - KL).sum()
        else:
            loss = -(RE - KL).mean()

        return loss

    def sample(self, batch_size=64):
        # This sample function defines how can we generate a new image
        
        # We let the z_T is sampled from standard Normal distributions
        z = torch.randn([batch_size, self.D])
        
        # we each step, we use reverse step to sequentially generate images
        for i in range(len(self.p_dnns) - 1, -1, -1):
            h = self.p_dnns[i](z)
            mu_i, log_var_i = torch.chunk(h, 2, dim=1)
            z = self.reparameterization(torch.tanh(mu_i), log_var_i)

        mu_x = self.decoder_net(z)

        return mu_x

    def sample_diffusion(self, x):
        
        # Alternatively, we can use sample_diffusion to sequentially add noise to images until it becomes white noise
        zs = [self.reparameterization_gaussian_diffusion(x, 0)]

        for i in range(1, self.T):
            zs.append(self.reparameterization_gaussian_diffusion(zs[-1], i))

        return zs[-1]

## Evaluation and Training functions

**Evaluation step, sampling and curve plotting**

In [28]:
def evaluation(test_loader, name=None, model_best=None, epoch=None):
    # EVALUATION
    if model_best is None:
        # load best performing model
        model_best = torch.load(name + '.model')

    model_best.eval()
    loss = 0.
    N = 0.
    for indx_batch, test_batch in enumerate(test_loader):
        loss_t = model_best.forward(test_batch, reduction='sum')
        loss = loss + loss_t.item()
        N = N + test_batch.shape[0]
    loss = loss / N

    if epoch is None:
        print(f'FINAL LOSS: nll={loss}')
    else:
        print(f'Epoch: {epoch}, val nll={loss}')

    return loss


def samples_real(name, test_loader):
    # REAL-------
    num_x = 4
    num_y = 4
    x = next(iter(test_loader)).detach().numpy()

    fig, ax = plt.subplots(num_x, num_y)
    for i, ax in enumerate(ax.flatten()):
        plottable_image = np.reshape(x[i], (8, 8))
        ax.imshow(plottable_image, cmap='gray')
        ax.axis('off')

    plt.savefig(name+'_real_images.pdf', bbox_inches='tight')
    plt.close()


def samples_generated(name, data_loader, extra_name=''):
    # GENERATIONS-------
    model_best = torch.load(name + '.model',weights_only=False)
    model_best.eval()

    num_x = 4
    num_y = 4
    
    # this model.sample() function is to sample new images    
    x = model_best.sample(batch_size=num_x * num_y)
    x = x.detach().numpy()

    fig, ax = plt.subplots(num_x, num_y)
    for i, ax in enumerate(ax.flatten()):
        plottable_image = np.reshape(x[i], (8, 8))
        ax.imshow(plottable_image, cmap='gray')
        ax.axis('off')

    plt.savefig(name + '_generated_images' + extra_name + '.pdf', bbox_inches='tight')
    plt.close()

    
def samples_diffusion(name, data_loader, extra_name=''):
    x = next(iter(data_loader))

    # GENERATIONS-------
    model_best = torch.load(name + '.model',weights_only=False)
    model_best.eval()

    num_x = 4
    num_y = 4
    z = model_best.sample_diffusion(x)
    z = z.detach().numpy()

    fig, ax = plt.subplots(num_x, num_y)
    for i, ax in enumerate(ax.flatten()):
        plottable_image = np.reshape(z[i], (8, 8))
        ax.imshow(plottable_image, cmap='gray')
        ax.axis('off')

    plt.savefig(name + '_generated_diffusion' + extra_name + '.pdf', bbox_inches='tight')
    plt.close()
    

def plot_curve(name, nll_val):
    plt.plot(np.arange(len(nll_val)), nll_val, linewidth='3')
    plt.xlabel('epochs')
    plt.ylabel('nll')
    plt.savefig(name + '_nll_val_curve.pdf', bbox_inches='tight')
    plt.close()

**Training step**

In [29]:
def training(name, max_patience, num_epochs, model, optimizer, training_loader, val_loader):
    nll_val = []
    best_nll = 1000.
    patience = 0

    # Main loop
    for e in range(num_epochs):
        # TRAINING
        model.train()
        for indx_batch, batch in enumerate(training_loader):
            loss = model.forward(batch)

            # this is the key step in pytorch to optimize neural network parameters
            optimizer.zero_grad()
            loss.backward(retain_graph=True)
            optimizer.step()

        # Validation
        loss_val = evaluation(val_loader, model_best=model, epoch=e)
        nll_val.append(loss_val)  # save for plotting

        if e == 0:
            print('saved!')
            torch.save(model, name + '.model')
            best_nll = loss_val
        else:
            if loss_val < best_nll:
                print('saved!')
                torch.save(model, name + '.model')
                best_nll = loss_val
                patience = 0

                samples_generated(name, val_loader, extra_name="_epoch_" + str(e))
            else:
                patience = patience + 1

        if patience > max_patience:
            break

    nll_val = np.asarray(nll_val)

    return nll_val

## Experiments

**Initialize datasets**

In [30]:
transforms = tt.Lambda(lambda x: 2. * (x / 17.) - 1.)

In [31]:
train_data = Digits(mode='train', transforms=transforms)
val_data = Digits(mode='val', transforms=transforms)
test_data = Digits(mode='test', transforms=transforms)

training_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

**Hyperparameters**

In [32]:
D = 64   # input dimension

M = 256  # the number of neurons in scale (s) and translation (t) nets

# the number of steps in DDPM. In real application, T can be set as large as 1,000 steps
T = 500

beta = 0.9

lr = 1e-3 # learning rate
num_epochs = 10 # max. number of epochs
max_patience = 50 # an early stopping is used, if training doesn't improve for longer than 20 epochs, it is stopped

**Creating a folder for results**

In [33]:
name = 'ddmg' + '_' + str(T) + '_' + str(beta)
# result_dir = 'results/' + name + '/'
result_dir = 'results/' 
if not (os.path.exists(result_dir)):
    os.mkdir(result_dir)

**Initializing the model**

In [34]:
# define the neural network in the denoise steps
p_dnns = [nn.Sequential(nn.Linear(D, M), nn.LeakyReLU(),
                        nn.Linear(M, M), nn.LeakyReLU(),
                        nn.Linear(M, M), nn.LeakyReLU(),
                        nn.Linear(M, 2 * D)) for _ in range(T-1)]

# define the neural network for reconstruct images
decoder_net = nn.Sequential(nn.Linear(D, M*2), nn.LeakyReLU(),
                            nn.Linear(M*2, M*2), nn.LeakyReLU(),
                            nn.Linear(M*2, M*2), nn.LeakyReLU(),
                            nn.Linear(M*2, D), nn.Tanh())


# Eventually, we initialize the full model
model = DDPM(p_dnns, decoder_net, beta=beta, T=T, D=D)

**Optimizer - here we use Adamax**

In [35]:
# OPTIMIZER
optimizer = torch.optim.Adamax([p for p in model.parameters() if p.requires_grad == True], lr=lr)

**Training loop**

In [36]:
# Training procedure
nll_val = training(name=result_dir + name, max_patience=max_patience, num_epochs=num_epochs, model=model, optimizer=optimizer,
                       training_loader=training_loader, val_loader=val_loader)

Epoch: 0, val nll=9515.959553571429
saved!
Epoch: 1, val nll=9518.143839285714
Epoch: 2, val nll=9514.276607142858
saved!


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.DDPM was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.DDPM])` or the `torch.serialization.safe_globals([__main__.DDPM])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weight

**The final evaluation**

In [15]:
test_loss = evaluation(name=result_dir + name, test_loader=test_loader)
f = open(result_dir + name + '_test_loss.txt', "w")
f.write(str(test_loss))
f.close()

samples_real(result_dir + name, test_loader)

plot_curve(result_dir + name, nll_val)

samples_generated(result_dir + name, test_loader, extra_name='FINAL')
samples_diffusion(result_dir + name, test_loader, extra_name='DIFFUSION')

FINAL LOSS: nll=160.1297866855425


***Other interesting materials about DDPM can be found in the following link***

https://github.com/lucidrains/denoising-diffusion-pytorch

****Task****

We have just shown a rather simple example to use DDPM. 

In real world situation, U-net, rather than the simple Multiple-Layer Perception (MLP), is used much more often.
In the following minutes, please use the below link to read line-by-line by yourself to understand the code, and run the code for more complex images.

https://colab.research.google.com/github/aju22/DDPM/blob/main/Denoising_Diffusion_Probabilistic_Model_(DDPMs).ipynb